# Production - Run All Scenarios for a Country and Update Heavy Crude Production

## Purpose
Mid-level orchestrator notebook that processes all production forecast scenarios for a single country, then calculates heavy crude production splits and updates annual aggregated forecasts.

## Orchestration Flow

[04-Production - Looping All Countries and Scenarios](#notebook-4214794422884444) → **This notebook** → [Production Forecast - Single Scenario by Country](#notebook-212233715354892)

### Position in Pipeline
This notebook is called by the top-level orchestrator for each country and:
1. Loops through all scenarios for the country
2. Calls the worker notebook to generate monthly forecasts for each scenario
3. Aggregates monthly forecasts into annual totals
4. Applies heavy crude percentage splits from Google Sheets
5. Projects future heavy crude percentages using 5-year CAGR
6. Calculates heavy crude volumes for historical and forecast periods

## Data Sources (Read)
* `workspace.gold.exogenous_variables` - Scenario definitions for the country
* `workspace.gold.production_forecast_table_monthly` - Monthly forecasts generated by worker notebook
* `workspace.gold.production_forecast_table_annual` - Existing annual data (for updates)
* Google Sheets - Heavy crude API percent splits by country and year

## Data Destinations (Write)
* `workspace.gold.production_forecast_table_annual` - Annual aggregated forecasts with heavy crude calculations
* `workspace.gold.production_forecast_table_monthly` - Monthly forecasts updated with heavy crude splits

## Workflow
1. **Receive country parameter** via dbutils widget (passed from parent orchestrator)
2. **Query scenarios** from `gold.exogenous_variables` for this country
3. **Loop through scenarios** and call worker notebook for each to generate monthly forecasts
4. **Aggregate to annual** - Sum monthly forecasts by year, excluding incomplete current year from historical totals
5. **Load heavy crude splits** from Google Sheets for this country
6. **Update annual table** with historical heavy crude percentages
7. **Project future percentages** using 5-year CAGR from most recent data
8. **Calculate heavy crude volumes** by multiplying total production by heavy crude percentage
9. **Apply splits to monthly table** - Update monthly forecasts with heavy crude calculations
10. **Verify generic names** - Check that scenarios have generic_name assigned in exogenous_variables
11. **Create long format table** - Unpivot exogenous_variables to create `exogenous_variables_long`
12. **Update forecast metadata** - Add generic_name and event_direction columns to annual and monthly forecast tables

## Key Features
* **Parameterized execution** - Receives country via widget, processes all scenarios for that country
* **Annual aggregation** - Converts monthly forecasts to annual totals with proper handling of incomplete years
* **Heavy crude modeling** - Applies API-based splits and projects future percentages
* **Connection points** - Copies last complete historical year to forecast columns for smooth visualization
* **Comprehensive bounds** - Calculates heavy crude confidence intervals (lower/upper bounds)

In [0]:
# Receive country parameter from widget (can be passed from other notebooks via dbutils.notebook.run)
dbutils.widgets.text("country", "")
country = dbutils.widgets.get("country")

print(f"Running all scenarios for: {country}")

#get all unique scenarios for the given country from workspace.gold.exogenous_variables
scenario_list_df = spark.sql(f"""
SELECT DISTINCT description, Event_Direction
FROM workspace.gold.exogenous_variables
WHERE country = '{country}' AND (Event_Direction = 'Production' OR Event_Direction = 'Baseline')
ORDER BY description
""")

# scenario_list_df.display()  # Commented out - causes hang when called via dbutils.notebook.run()

# Calculate total scenarios and estimated time
total_scenarios = scenario_list_df.count()
estimated_minutes = total_scenarios * 10  # Assuming 10 minutes per scenario

print(f"\n{country} has {total_scenarios} scenarios to run. Estimated time to run all scenarios is {estimated_minutes} minutes.")


Running all scenarios for: Brazil

Brazil has 1 scenarios to run. Estimated time to run all scenarios is 10 minutes.


#Run the Production Forecasts for the given country

In [0]:
#get all unique Production scenarios for the given country from workspace.gold.exogenous_variables
scenario_list_df = spark.sql(f"""
SELECT DISTINCT description
FROM workspace.gold.exogenous_variables
WHERE country = '{country}' and (Event_Direction = 'Production' or Event_Direction = 'Baseline')
ORDER BY description
""")

# scenario_list_df.display()  # Commented out - causes hang when called via dbutils.notebook.run()

#collect all scenarios into a list
scenarios = [row.description for row in scenario_list_df.collect()]
total_scenarios = len(scenarios)

print(f"\n{'='*60}")
print(f"Starting forecast loop for {country}")
print(f"Total scenarios to process: {total_scenarios}")
print(f"{'='*60}\n")

#loop through each scenario and run the forecast
for idx, scenario in enumerate(scenarios, start=1):
  print(f"\n[{idx}/{total_scenarios}] Processing scenario: {scenario}")
  print("-" * 60)
  
  #run the Production Forecast - Single Scenario by Country notebook for each scenario
  # timeout = 1800 seconds (30 minutes) to ensure the notebook completes before moving to next scenario
  dbutils.notebook.run(
    '/Workspace/Users/mijimorgan@gmail.com/Modelling Workflow Scripts/2-Forecasting/Worker Notebooks/Production Forecast - Single Scenario by Country', 
    1800, 
    {'scenario': scenario, 'country': country})
  
  print(f"✓ Completed {idx}/{total_scenarios} scenarios ({int(idx/total_scenarios*100)}% done)")

print(f"\n{'='*60}")
print(f"✓ ALL SCENARIOS COMPLETED for {country}")
print(f"Total scenarios processed: {total_scenarios}")
print(f"{'='*60}")




Starting forecast loop for Brazil
Total scenarios to process: 1


[1/1] Processing scenario: Baseline scenario
------------------------------------------------------------
✓ Completed 1/1 scenarios (100% done)

✓ ALL SCENARIOS COMPLETED for Brazil
Total scenarios processed: 1


#Update the annual forecast table


In [0]:
# Update production_forecast_table_annual for the country that was just processed
print(f"Updating production_forecast_table_annual for {country}...")

# First, delete existing annual data for this country
spark.sql(f"""
    DELETE FROM workspace.gold.production_forecast_table_annual
    WHERE Country = '{country}'
""")

print(f"  Deleted existing annual data for {country}")

# Now insert fresh annual aggregations from the monthly table
# IMPORTANT: Exclude the current year from Total_crude aggregation since it's incomplete
# Use conditional aggregation to handle current year vs historical years
spark.sql(f"""
    INSERT INTO workspace.gold.production_forecast_table_annual
    SELECT 
        Country,
        Scenario,
        DATE_TRUNC('YEAR', Date) as Date,
        SUM(CASE WHEN YEAR(Date) < YEAR(CURRENT_DATE()) THEN Total_crude ELSE NULL END) as Total_crude,
        SUM(Total_Lower_Bound) as Total_Lower_Bound,
        SUM(Total_Upper_Bound) as Total_Upper_Bound,
        NULL as Heavy_crude,
        NULL as Heavy_Lower_Bound,
        NULL as Heavy_Upper_Bound,
        NULL as Heavy_Crude_Percent,
        SUM(Total_crude_forecast) as Total_crude_forecast,
        NULL as Heavy_crude_forecast,
        NULL as generic_name,
        NULL as event_direction
    FROM workspace.gold.production_forecast_table_monthly
    WHERE Country = '{country}'
    GROUP BY Country, Scenario, DATE_TRUNC('YEAR', Date)
    ORDER BY Country, Scenario, Date
""")

print(f"✓ Annual forecast table updated for {country}")

# Add connection point: Copy last complete historical year's Total_crude to Total_crude_forecast
print(f"\nAdding connection point (last historical year in forecast columns)...")
spark.sql(f"""
    UPDATE workspace.gold.production_forecast_table_annual
    SET Total_crude_forecast = Total_crude
    WHERE Country = '{country}'
      AND YEAR(Date) = YEAR(CURRENT_DATE()) - 1
      AND Total_crude IS NOT NULL
""")

connection_count = spark.sql(f"""
    SELECT COUNT(*) as count
    FROM workspace.gold.production_forecast_table_annual
    WHERE Country = '{country}'
      AND Total_crude IS NOT NULL
      AND Total_crude_forecast IS NOT NULL
""").collect()[0]['count']

print(f"✓ Created {connection_count} connection points (last historical year in each scenario)")

# Show summary of what was inserted
annual_summary = spark.sql(f"""
    SELECT 
        Country,
        COUNT(DISTINCT Scenario) as num_scenarios,
        COUNT(*) as total_annual_records,
        MIN(YEAR(Date)) as earliest_year,
        MAX(YEAR(Date)) as latest_year
    FROM workspace.gold.production_forecast_table_annual
    WHERE Country = '{country}'
    GROUP BY Country
""")

annual_summary.display()

print(f"\nAnnual aggregation complete for {country}")

Updating production_forecast_table_annual for Brazil...
  Deleted existing annual data for Brazil
✓ Annual forecast table updated for Brazil

Adding connection point (last historical year in forecast columns)...
✓ Created 1 connection points (last historical year in each scenario)


Country,num_scenarios,total_annual_records,earliest_year,latest_year
Brazil,1,48,1984,2031



Annual aggregation complete for Brazil


#Update the heavy crude percent splits and calculate new heavy crude numbers

In [0]:
# import libraries
import pandas as pd
import sklearn
import numpy as np

#collect the % split assumptions
sheet_id = "1lf7Qd0MjLfp8C7h5HZuPlOXIvYdlDHUyUYsMUOPoufw"
gid_p = "947317138" # sheet tab ID

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid_p}"

# Load CSVs via pandas first (Serverless doesn't support direct HTTP reads with Spark)
df = spark.createDataFrame(pd.read_csv(url))

# Clean column names (Delta doesn't allow special characters)
def clean_column_names(df):
    for col in df.columns:
        clean_col = col.replace('(', '').replace(')', '').replace(',', '').replace(' ', '_').replace('/','per')
        if col != clean_col:
            df = df.withColumnRenamed(col, clean_col)
    return df

df = clean_column_names(df)


In [0]:
#collect the splits
df_splits = df.filter(df['Metric'] == 'Percent Heavy by API')

#pivot the table. Unpivot all year columns into Year and Percent columns
from pyspark.sql.functions import col, expr, year as year_func

# Get all year columns (assuming they're numeric columns representing years)
year_columns = [c for c in df_splits.columns if c.isdigit() and int(c) >= 1965]

# Unpivot: stack all year columns into (Year, Percent) pairs
df_unpivoted = df_splits.selectExpr(
    "Country",
    f"stack({len(year_columns)}, {', '.join([f"'{yr}', `{yr}`" for yr in year_columns])}) as (Year, Heavy_Crude_Percent)"
)

# Convert Year to integer and Percent to double
df_unpivoted = df_unpivoted.withColumn("Year", col("Year").cast("int")) \
                           .withColumn("Heavy_Crude_Percent", col("Heavy_Crude_Percent").cast("double"))

# Filter for the current country
df_country_splits = df_unpivoted.filter(col("Country") == country)

print(f"Loaded Heavy Crude percent splits for {country}")
df_country_splits.display()

# Update the production_forecast_table_annual with the Heavy_Crude_Percent values
print(f"\nUpdating Heavy_Crude_Percent in production_forecast_table_annual for {country}...")

# Create a temp view for the splits data
df_country_splits.createOrReplaceTempView("temp_heavy_splits")

# Update the annual table by merging with the splits
spark.sql(f"""
    MERGE INTO workspace.gold.production_forecast_table_annual AS target
    USING (
        SELECT 
            '{country}' as Country,
            Year,
            Heavy_Crude_Percent
        FROM temp_heavy_splits
    ) AS source
    ON target.Country = source.Country 
       AND YEAR(target.Date) = source.Year
    WHEN MATCHED THEN UPDATE SET
        target.Heavy_Crude_Percent = source.Heavy_Crude_Percent
""")

print(f"✓ Heavy_Crude_Percent updated for {country}")



Loaded Heavy Crude percent splits for Brazil


Country,Year,Heavy_Crude_Percent
Brazil,1965,0.5
Brazil,1966,0.5
Brazil,1967,0.5
Brazil,1968,0.5
Brazil,1969,0.5
Brazil,1970,0.5
Brazil,1971,0.5
Brazil,1972,0.5
Brazil,1973,0.5
Brazil,1974,0.5



Updating Heavy_Crude_Percent in production_forecast_table_annual for Brazil...
✓ Heavy_Crude_Percent updated for Brazil


# Calculate future percent splits based off of average 5 year growth rate

In [0]:
# Calculate future Heavy_Crude_Percent values using 5-year average growth rate
from pyspark.sql.functions import col, max as spark_max, min as spark_min, year as year_func

print(f"Calculating future Heavy_Crude_Percent values for {country}...\n")

# Get the current state of the annual table for this country
df_annual = spark.sql(f"""
    SELECT 
        Country,
        Scenario,
        YEAR(Date) as Year,
        Total_crude,
        Heavy_Crude_Percent
    FROM workspace.gold.production_forecast_table_annual
    WHERE Country = '{country}'
    ORDER BY Year
""")

# Find the last year with Heavy_Crude_Percent data and the last year with Total_crude data
last_heavy_year = df_annual.filter(col("Heavy_Crude_Percent").isNotNull()) \
                            .agg(spark_max("Year").alias("year")).collect()[0]["year"]

last_total_year = df_annual.agg(spark_max("Year").alias("year")).collect()[0]["year"]

print(f"Last year with Heavy_Crude_Percent data: {last_heavy_year}")
print(f"Last year with Total_crude data: {last_total_year}")

if last_total_year <= last_heavy_year:
    print(f"\n✓ No future projection needed - Heavy_Crude_Percent already covers all years with Total_crude data")
else:
    # Reset any existing projected values (years after the last source data year)
    print(f"\nResetting projected values for years after {last_heavy_year}...")
    spark.sql(f"""
        UPDATE workspace.gold.production_forecast_table_annual
        SET Heavy_Crude_Percent = NULL
        WHERE Country = '{country}' AND YEAR(Date) > {last_heavy_year}
    """)
    print(f"✓ Reset complete - ready to recalculate projections\n")
    # Get the last 5 years of Heavy_Crude_Percent data
    df_last_5_years = df_annual.filter(
        (col("Year") <= last_heavy_year) & 
        (col("Year") > last_heavy_year - 5) &
        (col("Heavy_Crude_Percent").isNotNull())
    ).groupBy("Year").agg(spark_max("Heavy_Crude_Percent").alias("Heavy_Crude_Percent")).orderBy("Year")
    
    print(f"\nLast 5 years of Heavy_Crude_Percent data:")
    df_last_5_years.display()
    
    # Calculate CAGR: ((End Value / Start Value)^(1/years)) - 1
    last_5_data = df_last_5_years.collect()
    
    if len(last_5_data) >= 2:
        start_value = last_5_data[0]["Heavy_Crude_Percent"]
        end_value = last_5_data[-1]["Heavy_Crude_Percent"]
        start_year = last_5_data[0]["Year"]
        end_year = last_5_data[-1]["Year"]
        years_diff = end_year - start_year
        
        if start_value > 0 and years_diff > 0:
            avg_growth = (end_value - start_value) / years_diff
            
            print(f"\nCalculated 5-year average growth rate: {avg_growth:.6f} percentage points per year")
            print(f"  Start: {start_value:.4f} ({start_year})")
            print(f"  End: {end_value:.4f} ({end_year})")
            print(f"  Change: {end_value - start_value:.4f} over {years_diff} years")
            
            # Generate future values
            future_years = list(range(last_heavy_year + 1, last_total_year + 1))
            future_values = []
            
            current_value = end_value
            print(f"\nProjecting forward from {last_heavy_year} to {last_total_year}:")
            
            for year in future_years:
                years_forward = year - end_year
                projected_value = end_value + (avg_growth * years_forward)
                # Floor at 0 to prevent negative percentages
                projected_value = max(0, projected_value)
                future_values.append((country, year, projected_value))
                print(f"  {year}: {projected_value:.4f}")
            
            # Create DataFrame with future projections
            df_future = spark.createDataFrame(future_values, ["Country", "Year", "Heavy_Crude_Percent"])
            
            # Update the production_forecast_table_annual with projected values
            print(f"\nUpdating production_forecast_table_annual with projected Heavy_Crude_Percent values...")
            
            df_future.createOrReplaceTempView("temp_future_heavy_splits")
            
            spark.sql(f"""
                MERGE INTO workspace.gold.production_forecast_table_annual AS target
                USING (
                    SELECT 
                        Country,
                        Year,
                        Heavy_Crude_Percent
                    FROM temp_future_heavy_splits
                ) AS source
                ON target.Country = source.Country 
                   AND YEAR(target.Date) = source.Year
                WHEN MATCHED THEN UPDATE SET
                    target.Heavy_Crude_Percent = source.Heavy_Crude_Percent
            """)
            
            print(f"✓ Projected Heavy_Crude_Percent values updated for {country}")
            
            # Verify the complete update
            verify_df = spark.sql(f"""
                SELECT 
                    YEAR(Date) as Year,
                    AVG(Heavy_Crude_Percent) as Heavy_Crude_Percent,
                    COUNT(*) as num_scenarios
                FROM workspace.gold.production_forecast_table_annual
                WHERE Country = '{country}'
                GROUP BY YEAR(Date)
                ORDER BY Year DESC
                LIMIT 20
            """)
            
            print(f"\nLast 20 years of Heavy_Crude_Percent (most recent first):")
            verify_df.display()
            
        else:
            print(f"\n⚠ Cannot calculate CAGR - invalid start value or years difference")
    else:
        print(f"\n⚠ Not enough data points to calculate 5-year CAGR (need at least 2 years)")

print(f"\n✓ Heavy crude percent projection complete for {country}")

Calculating future Heavy_Crude_Percent values for Brazil...

Last year with Heavy_Crude_Percent data: 2024
Last year with Total_crude data: 2031

Resetting projected values for years after 2024...
✓ Reset complete - ready to recalculate projections


Last 5 years of Heavy_Crude_Percent data:


Year,Heavy_Crude_Percent
2020,0.35
2021,0.35
2022,0.35
2023,0.35
2024,0.35



Calculated 5-year average growth rate: 0.000000 percentage points per year
  Start: 0.3500 (2020)
  End: 0.3500 (2024)
  Change: 0.0000 over 4 years

Projecting forward from 2024 to 2031:
  2025: 0.3500
  2026: 0.3500
  2027: 0.3500
  2028: 0.3500
  2029: 0.3500
  2030: 0.3500
  2031: 0.3500

Updating production_forecast_table_annual with projected Heavy_Crude_Percent values...
✓ Projected Heavy_Crude_Percent values updated for Brazil

Last 20 years of Heavy_Crude_Percent (most recent first):


Year,Heavy_Crude_Percent,num_scenarios
2031,0.35,1
2030,0.35,1
2029,0.35,1
2028,0.35,1
2027,0.35,1
2026,0.35,1
2025,0.35,1
2024,0.35,1
2023,0.35,1
2022,0.35,1



✓ Heavy crude percent projection complete for Brazil


#Update heavy_crude values, historical and forecast

In [0]:
# Calculate Heavy_crude values by multiplying Total columns by Heavy_Crude_Percent
print(f"Calculating Heavy crude values for {country}...\n")

spark.sql(f"""
    UPDATE workspace.gold.production_forecast_table_annual
    SET 
        Heavy_crude = CASE WHEN Total_crude IS NOT NULL THEN GREATEST(0, Total_crude * (Heavy_Crude_Percent)) ELSE NULL END,
        Heavy_Lower_Bound = CASE WHEN Total_Lower_Bound IS NOT NULL THEN GREATEST(0, Total_Lower_Bound * (Heavy_Crude_Percent)) ELSE NULL END,
        Heavy_Upper_Bound = CASE WHEN Total_Upper_Bound IS NOT NULL THEN GREATEST(0, Total_Upper_Bound * (Heavy_Crude_Percent)) ELSE NULL END,
        Heavy_crude_forecast = CASE WHEN Total_crude_forecast IS NOT NULL THEN GREATEST(0, Total_crude_forecast * (Heavy_Crude_Percent)) ELSE NULL END
    WHERE Country = '{country}'
      AND Heavy_Crude_Percent IS NOT NULL
""")

print(f"✓ Heavy crude values calculated for {country}\n")

# Verify the calculations
print("Verification - Sample of updated records (most recent years):")
verify_df = spark.sql(f"""
    SELECT 
        Country,
        Scenario,
        YEAR(Date) as Year,
        ROUND(Total_crude, 2) as Total_crude,
        ROUND(Heavy_Crude_Percent, 2) as Heavy_Crude_Percent,
        ROUND(Heavy_crude, 2) as Heavy_crude,
        ROUND(Heavy_Lower_Bound, 2) as Heavy_Lower_Bound,
        ROUND(Heavy_Upper_Bound, 2) as Heavy_Upper_Bound,
        ROUND(Heavy_crude, 2) as Heavy_crude,
        ROUND(Total_crude_forecast, 2) as Total_crude_forecast,
        ROUND(Heavy_crude_forecast, 2) as Heavy_crude_forecast
    FROM workspace.gold.production_forecast_table_annual
    WHERE Country = '{country}'
      AND Scenario = 'baseline_scenario'
    ORDER BY Year DESC
    LIMIT 10
""")

verify_df.display()

print(f"\n✓ Heavy crude calculation complete for {country}")

Calculating Heavy crude values for Brazil...

✓ Heavy crude values calculated for Brazil

Verification - Sample of updated records (most recent years):


Country,Scenario,Year,Total_crude,Heavy_Crude_Percent,Heavy_crude,Heavy_Lower_Bound,Heavy_Upper_Bound,Heavy_crude,Total_crude_forecast,Heavy_crude_forecast
Brazil,baseline_scenario,2031,null,0.35,null,2676.98,3546.63,null,8890.86,3111.8
Brazil,baseline_scenario,2030,null,0.35,null,16003.37,20915.77,null,52741.63,18459.57
Brazil,baseline_scenario,2029,null,0.35,null,15926.05,20268.83,null,51706.97,18097.44
Brazil,baseline_scenario,2028,null,0.35,null,15892.59,19578.03,null,50672.31,17735.31
Brazil,baseline_scenario,2027,null,0.35,null,15933.58,18812.78,null,49637.66,17373.18
Brazil,baseline_scenario,2026,null,0.35,null,13446.53,14970.4,null,44656.63,15629.82
Brazil,baseline_scenario,2025,45205.56,0.35,15821.94,null,null,15821.94,45205.56,15821.94
Brazil,baseline_scenario,2024,40282.16,0.35,14098.76,null,null,14098.76,null,null
Brazil,baseline_scenario,2023,40813.67,0.35,14284.78,null,null,14284.78,null,null
Brazil,baseline_scenario,2022,36247.42,0.35,12686.6,null,null,12686.6,null,null



✓ Heavy crude calculation complete for Brazil


# Apply splits to the monthly forecast and create a monthly heavy crude forecast

In [0]:
# Apply splits to the monthly production forecast to display a monthly heavy crude forecast
# The annual split will be applied to all months in the year
# Then calculate heavy crude for each month by multiplying the monthly total by the monthly heavy crude percentage

print(f"Applying Heavy_Crude_Percent splits to monthly forecast for {country}...\n")

# First, check if the columns exist in the monthly table, if not add them
monthly_schema = spark.table('workspace.gold.production_forecast_table_monthly').schema
column_names = [field.name for field in monthly_schema.fields]

if 'Heavy_Crude_Percent' not in column_names:
    print("Adding Heavy_Crude_Percent column to monthly table...")
    spark.sql("""
        ALTER TABLE workspace.gold.production_forecast_table_monthly 
        ADD COLUMN Heavy_Crude_Percent DOUBLE
    """)
    print("✓ Heavy_Crude_Percent column added")

if 'Heavy_crude' not in column_names:
    print("Adding Heavy_crude column to monthly table...")
    spark.sql("""
        ALTER TABLE workspace.gold.production_forecast_table_monthly 
        ADD COLUMN Heavy_crude DOUBLE
    """)
    print("✓ Heavy_crude column added")

if 'Heavy_crude_forecast' not in column_names:
    print("Adding Heavy_crude_forecast column to monthly table...")
    spark.sql("""
        ALTER TABLE workspace.gold.production_forecast_table_monthly 
        ADD COLUMN Heavy_crude_forecast DOUBLE
    """)
    print("✓ Heavy_crude_forecast column added")

if 'Heavy_Lower_Bound' not in column_names:
    print("Adding Heavy_Lower_Bound column to monthly table...")
    spark.sql("""
        ALTER TABLE workspace.gold.production_forecast_table_monthly 
        ADD COLUMN Heavy_Lower_Bound DOUBLE
    """)
    print("✓ Heavy_Lower_Bound column added")

if 'Heavy_Upper_Bound' not in column_names:
    print("Adding Heavy_Upper_Bound column to monthly table...")
    spark.sql("""
        ALTER TABLE workspace.gold.production_forecast_table_monthly 
        ADD COLUMN Heavy_Upper_Bound DOUBLE
    """)
    print("✓ Heavy_Upper_Bound column added\n")

# Update the monthly table with Heavy_Crude_Percent from the annual table
# Join on Country, Scenario, and Year
print(f"Updating Heavy_Crude_Percent for {country} in monthly table...")
spark.sql(f"""
    MERGE INTO workspace.gold.production_forecast_table_monthly AS monthly
    USING (
        SELECT 
            Country,
            Scenario,
            YEAR(Date) as Year,
            Heavy_Crude_Percent
        FROM workspace.gold.production_forecast_table_annual
        WHERE Country = '{country}'
          AND Heavy_Crude_Percent IS NOT NULL
    ) AS annual
    ON monthly.Country = annual.Country 
       AND monthly.Scenario = annual.Scenario
       AND YEAR(monthly.Date) = annual.Year
    WHEN MATCHED THEN UPDATE SET
        monthly.Heavy_Crude_Percent = annual.Heavy_Crude_Percent
""")

print(f"✓ Heavy_Crude_Percent updated for {country}\n")

# Calculate Heavy_crude, Heavy_crude_forecast, and Heavy bounds
print(f"Calculating Heavy_crude values and bounds for {country}...")
spark.sql(f"""
    UPDATE workspace.gold.production_forecast_table_monthly
    SET 
        Heavy_crude = CASE WHEN Total_crude IS NOT NULL THEN GREATEST(0, Total_crude * (Heavy_Crude_Percent)) ELSE NULL END,
        Heavy_crude_forecast = CASE WHEN Total_crude_forecast IS NOT NULL THEN GREATEST(0, Total_crude_forecast * (Heavy_Crude_Percent)) ELSE NULL END,
        Heavy_Lower_Bound = CASE WHEN Total_Lower_Bound IS NOT NULL THEN GREATEST(0, Total_Lower_Bound * (Heavy_Crude_Percent)) ELSE NULL END,
        Heavy_Upper_Bound = CASE WHEN Total_Upper_Bound IS NOT NULL THEN GREATEST(0, Total_Upper_Bound * (Heavy_Crude_Percent)) ELSE NULL END
    WHERE Country = '{country}'
      AND Heavy_Crude_Percent IS NOT NULL
""")

print(f"✓ Heavy_crude values and bounds calculated for {country}\n")

# Verify the results
print("Verification - Sample of updated monthly records (most recent months):")
verify_df = spark.sql(f"""
    SELECT 
        Country,
        Scenario,
        Date,
        ROUND(Total_crude, 2) as Total_crude,
        ROUND(Total_crude_forecast, 2) as Total_crude_forecast,
        ROUND(Total_Lower_Bound, 2) as Total_Lower_Bound,
        ROUND(Total_Upper_Bound, 2) as Total_Upper_Bound,
        ROUND(Heavy_Crude_Percent, 2) as Heavy_Crude_Percent,
        ROUND(Heavy_crude, 2) as Heavy_crude,
        ROUND(Heavy_crude_forecast, 2) as Heavy_crude_forecast,
        ROUND(Heavy_Lower_Bound, 2) as Heavy_Lower_Bound,
        ROUND(Heavy_Upper_Bound, 2) as Heavy_Upper_Bound
    FROM workspace.gold.production_forecast_table_monthly
    WHERE Country = '{country}'
      AND Scenario = 'baseline_scenario'
    ORDER BY Date DESC
    LIMIT 100
""")

verify_df.display()

print(f"\n✓ Monthly heavy crude calculation complete for {country}")



Applying Heavy_Crude_Percent splits to monthly forecast for Brazil...

Updating Heavy_Crude_Percent for Brazil in monthly table...
✓ Heavy_Crude_Percent updated for Brazil

Calculating Heavy_crude values and bounds for Brazil...
✓ Heavy_crude values and bounds calculated for Brazil

Verification - Sample of updated monthly records (most recent months):


Country,Scenario,Date,Total_crude,Total_crude_forecast,Total_Lower_Bound,Total_Upper_Bound,Heavy_Crude_Percent,Heavy_crude,Heavy_crude_forecast,Heavy_Lower_Bound,Heavy_Upper_Bound
Brazil,baseline_scenario,2031-02-01T00:00:00.000Z,null,4449.02,3825.34,5072.71,0.35,null,1557.16,1338.87,1775.45
Brazil,baseline_scenario,2031-01-01T00:00:00.000Z,null,4441.84,3823.16,5060.52,0.35,null,1554.64,1338.11,1771.18
Brazil,baseline_scenario,2030-12-01T00:00:00.000Z,null,4434.65,3821.02,5048.28,0.35,null,1552.13,1337.36,1766.9
Brazil,baseline_scenario,2030-11-01T00:00:00.000Z,null,4427.47,3818.93,5036.01,0.35,null,1549.61,1336.62,1762.6
Brazil,baseline_scenario,2030-10-01T00:00:00.000Z,null,4420.28,3816.87,5023.69,0.35,null,1547.1,1335.91,1758.29
Brazil,baseline_scenario,2030-09-01T00:00:00.000Z,null,4413.1,3814.86,5011.33,0.35,null,1544.58,1335.2,1753.97
Brazil,baseline_scenario,2030-08-01T00:00:00.000Z,null,4405.91,3812.9,4998.93,0.35,null,1542.07,1334.51,1749.63
Brazil,baseline_scenario,2030-07-01T00:00:00.000Z,null,4398.73,3810.98,4986.48,0.35,null,1539.55,1333.84,1745.27
Brazil,baseline_scenario,2030-06-01T00:00:00.000Z,null,4391.54,3809.11,4973.98,0.35,null,1537.04,1333.19,1740.89
Brazil,baseline_scenario,2030-05-01T00:00:00.000Z,null,4384.36,3807.29,4961.43,0.35,null,1534.53,1332.55,1736.5



✓ Monthly heavy crude calculation complete for Brazil


In [0]:
# Verify generic_name for the current country
# generic_name is now pre-computed in exogenous_variables (by Update Binary Scenario Variables notebook)
# This cell just displays the generic names that are already assigned

print(f"\nVerifying generic_name assignments for {country}...\n")

# Get all scenarios with generic_name for the current country
result = spark.sql(f"""
    SELECT Country, Event_Type, Event_Direction, Description, Coefficient, generic_name
    FROM workspace.gold.exogenous_variables
    WHERE Country = '{country}'
      AND generic_name IS NOT NULL
    ORDER BY Event_Type
""")

scenario_count = result.count()

if scenario_count > 0:
    print(f"✓ Found {scenario_count} scenarios with generic_name assigned for {country}")
    print(f"\nScenarios with generic names for {country}:")
    display(result)
else:
    print(f"⚠ WARNING: No scenarios with generic_name found for {country}")
    print(f"\nMake sure to run 'Update Binary Scenario Variables' notebook first to assign generic names.")


Verifying generic_name assignments for Brazil...

⚠ WARNING: No scenarios with generic_name found for Brazil

Make sure to run 'Update Binary Scenario Variables' notebook first to assign generic names.


In [0]:
# Load the exogenous_variables table
exo_table = spark.table('workspace.gold.exogenous_variables')

print(f"Original table has {exo_table.count()} rows")
print(f"Columns: {len(exo_table.columns)}")

# Identify metadata columns vs date columns
metadata_cols = ['Country', 'Event_Direction', 'Event_Type', 'Description', 'Coefficient', 'generic_name']
date_cols = [col for col in exo_table.columns if col not in metadata_cols]

print(f"\nMetadata columns: {len(metadata_cols)}")
print(f"Date columns to unpivot: {len(date_cols)} (from {date_cols[0]} to {date_cols[-1]})")

# Unpivot using stack() - creates one row per date per event
from pyspark.sql.functions import expr, col, to_date

# Build the stack expression: stack(n, 'col1', col1, 'col2', col2, ...)
stack_expr = f"stack({len(date_cols)}, {', '.join([f"'{c}', `{c}`" for c in date_cols])}) as (Date, Value)"

print(f"\nUnpivoting {len(date_cols)} date columns...")

# Apply stack to unpivot
exo_long = exo_table.select(
    *metadata_cols,
    expr(stack_expr)
)

# Convert Date string (YYYY-MM) to actual date type (first day of month)
from pyspark.sql.functions import concat, lit
exo_long = exo_long.withColumn('Date', to_date(concat(col('Date'), lit('-01')), 'yyyy-MM-dd'))

# Rename Value to Binary_Value for clarity
exo_long = exo_long.withColumnRenamed('Value', 'Binary_Value')

print(f"\n✓ Unpivoted table created with {exo_long.count()} rows")

# Show sample
print("\nSample of unpivoted data:")
display(exo_long.orderBy('Country', 'Description', 'Date').limit(20))

# Save to new table
print(f"\nSaving to workspace.gold.exogenous_variables_long...")
# Use mergeSchema to handle schema evolution (Event_Type column was added)
exo_long.write.mode('overwrite').option('mergeSchema', 'true').saveAsTable('workspace.gold.exogenous_variables_long')

print("\n" + "="*60)
print("✓ TABLE CREATED SUCCESSFULLY")
print("="*60)
print(f"Table: workspace.gold.exogenous_variables_long")
print(f"Total rows: {exo_long.count():,}")
print(f"Date range: {date_cols[0]} to {date_cols[-1]}")
print("\nSchema:")
print("  - Country (string)")
print("  - Event_Direction (string)")
print("  - Event_Type (string)")
print("  - Description (string)")
print("  - Coefficient (double)")
print("  - generic_name (string)")
print("  - Date (date)")
print("  - Binary_Value (int)")
print("="*60)

Original table has 63 rows
Columns: 644

Metadata columns: 6
Date columns to unpivot: 638 (from 1973-01 to 2026-02)

Unpivoting 638 date columns...

✓ Unpivoted table created with 40194 rows

Sample of unpivoted data:


Country,Event_Direction,Event_Type,Description,Coefficient,generic_name,Date,Binary_Value
Algeria,Baseline,null,Baseline scenario,null,null,1973-01-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-02-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-03-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-04-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-05-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-06-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-07-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-08-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-09-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-10-01,0



Saving to workspace.gold.exogenous_variables_long...

✓ TABLE CREATED SUCCESSFULLY
Table: workspace.gold.exogenous_variables_long
Total rows: 40,194
Date range: 1973-01 to 2026-02

Schema:
  - Country (string)
  - Event_Direction (string)
  - Event_Type (string)
  - Description (string)
  - Coefficient (double)
  - generic_name (string)
  - Date (date)
  - Binary_Value (int)


In [0]:
# Add generic_name to forecast tables by copying from exogenous_variables
# Join on: Country + Scenario (Description) + Event_Direction (Production/Consumption)

print(f"Adding generic_name to forecast tables for {country}...\n")

# List of tables to update
tables = [
    'workspace.gold.production_forecast_table_annual',
    'workspace.gold.production_forecast_table_monthly',
    'workspace.gold.consumption_forecast_table_annual'
]

for table_name in tables:
    print(f"Processing {table_name}...")
    
    # Check if generic_name column exists, if not add it
    table_schema = spark.table(table_name).schema
    column_names = [field.name for field in table_schema.fields]
    
    if 'generic_name' not in column_names:
        print(f"  Adding generic_name column...")
        spark.sql(f"""
            ALTER TABLE {table_name}
            ADD COLUMN generic_name STRING
        """)
        print(f"  ✓ Column added")
    else:
        print(f"  ✓ Column already exists")
    
    # Determine Event_Direction based on table name
    if 'production' in table_name:
        event_direction = 'Production'
    elif 'consumption' in table_name:
        event_direction = 'Consumption'
    else:
        event_direction = None
    
    # Update generic_name by joining with exogenous_variables
    # Match on Country + Scenario (Description in exogenous_variables) + Event_Direction
    print(f"  Updating generic_name values...")
    
    spark.sql(f"""
        MERGE INTO {table_name} AS forecast
        USING (
            SELECT DISTINCT
                Country,
                LOWER(REPLACE(Description, ' ', '_')) as Scenario_normalized,
                generic_name
            FROM workspace.gold.exogenous_variables
            WHERE Country = '{country}'
              AND Event_Direction = '{event_direction}'
              AND generic_name IS NOT NULL
        ) AS exo
        ON forecast.Country = exo.Country
           AND LOWER(REPLACE(forecast.Scenario, ' ', '_')) = exo.Scenario_normalized
        WHEN MATCHED THEN UPDATE SET
            forecast.generic_name = exo.generic_name
    """)
    
    # Verify the update
    verify_df = spark.sql(f"""
        SELECT 
            Scenario,
            generic_name,
            COUNT(*) as row_count
        FROM {table_name}
        WHERE Country = '{country}'
          AND generic_name IS NOT NULL
        GROUP BY Scenario, generic_name
        ORDER BY Scenario
    """)
    
    updated_count = verify_df.count()
    
    if updated_count > 0:
        print(f"  ✓ Updated {updated_count} unique scenario(s)")
        print(f"  Scenarios with generic names:")
        for row in verify_df.collect():
            print(f"    - {row.Scenario} → {row.generic_name} ({row.row_count} rows)")
    else:
        print(f"  ℹ No scenarios matched for update")
    
    print()

print("="*60)
print(f"✓ GENERIC_NAME UPDATE COMPLETE for {country}")
print("="*60)


Adding generic_name to forecast tables for Brazil...

Processing workspace.gold.production_forecast_table_annual...
  ✓ Column already exists
  Updating generic_name values...
  ℹ No scenarios matched for update

Processing workspace.gold.production_forecast_table_monthly...
  ✓ Column already exists
  Updating generic_name values...
  ℹ No scenarios matched for update

Processing workspace.gold.consumption_forecast_table_annual...
  ✓ Column already exists
  Updating generic_name values...
  ℹ No scenarios matched for update

✓ GENERIC_NAME UPDATE COMPLETE for Brazil


In [0]:
# Add event_direction to forecast tables
# For production notebook: 'baseline' if baseline_scenario, otherwise 'Production'

print(f"Adding event_direction to forecast tables for {country}...\n")

# List of production tables to update
tables = [
    'workspace.gold.production_forecast_table_annual',
    'workspace.gold.production_forecast_table_monthly'
]

for table_name in tables:
    print(f"Processing {table_name}...")
    
    # Check if event_direction column exists, if not add it
    table_schema = spark.table(table_name).schema
    column_names = [field.name for field in table_schema.fields]
    
    if 'event_direction' not in column_names:
        print(f"  Adding event_direction column...")
        spark.sql(f"""
            ALTER TABLE {table_name}
            ADD COLUMN event_direction STRING
        """)
        print(f"  ✓ Column added")
    else:
        print(f"  ✓ Column already exists")
    
    # Update event_direction based on scenario
    # baseline_scenario → 'baseline', all others → 'Production'
    print(f"  Updating event_direction values...")
    
    spark.sql(f"""
        UPDATE {table_name}
        SET event_direction = CASE 
            WHEN Scenario = 'baseline_scenario' THEN 'baseline'
            ELSE 'Production'
        END
        WHERE Country = '{country}'
    """)
    
    # Verify the update
    verify_df = spark.sql(f"""
        SELECT 
            Scenario,
            event_direction,
            COUNT(*) as row_count
        FROM {table_name}
        WHERE Country = '{country}'
        GROUP BY Scenario, event_direction
        ORDER BY Scenario
    """)
    
    print(f"  ✓ Updated {verify_df.count()} unique scenario(s)")
    print(f"  Event directions assigned:")
    for row in verify_df.collect():
        print(f"    - {row.Scenario} → {row.event_direction} ({row.row_count} rows)")
    
    print()

print("="*60)
print(f"✓ EVENT_DIRECTION UPDATE COMPLETE for {country}")
print("="*60)

Adding event_direction to forecast tables for Brazil...

Processing workspace.gold.production_forecast_table_annual...
  ✓ Column already exists
  Updating event_direction values...
  ✓ Updated 1 unique scenario(s)
  Event directions assigned:
    - baseline_scenario → baseline (48 rows)

Processing workspace.gold.production_forecast_table_monthly...
  ✓ Column already exists
  Updating event_direction values...
  ✓ Updated 1 unique scenario(s)
  Event directions assigned:
    - baseline_scenario → baseline (566 rows)

✓ EVENT_DIRECTION UPDATE COMPLETE for Brazil
